# backprop-pop-outgrad-loop composite — cx20: reverse-pass driver: pop next out-grad, dispatch back_fn from recipe

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `backprop-pop-outgrad-loop`, `dispatch-back-fn-from-recipe`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "backprop-pop-outgrad-loop"
DD_ATOM_IDS = ["backprop-pop-outgrad-loop", "dispatch-back-fn-from-recipe"]
DD_SUBTOPICS = ["Backprop: backprop pop-outgrad loop", "Backprop: dispatch back fn from recipe"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Composing the pop-outgrad loop with recipe-based dispatch

The reverse-pass driver is two atoms wired into one tight loop:

- **`backprop-pop-outgrad-loop`** — the OUTER walk: iterate the sorted
  graph (end-node first), pop each node's accumulated grad out of the
  `grads` dict, route to either leaf-write or parent-dispatch.
- **`dispatch-back-fn-from-recipe`** — the INNER step: for each parent,
  look up `back_funcs[(recipe.func, argnum)]` and call it.

```python
def backprop(end_node, end_grad, sorted_graph, back_funcs):
    grads = {id(end_node): end_grad}
    for node in sorted_graph:                       # pop-outgrad loop
        if id(node) not in grads: continue
        grad_out = grads.pop(id(node))              # POP, don't peek
        if node.recipe is None:                     # leaf — write .grad
            node.grad = grad_out if node.grad is None else node.grad + grad_out
            continue
        for argnum, parent in node.recipe.parents.items():    # dispatch loop
            back_fn = back_funcs[(node.recipe.func, argnum)]  # (atom 2)
            gp = back_fn(grad_out, node.array,
                         *node.recipe.args, **node.recipe.kwargs)
            grads[id(parent)] = grads.get(id(parent), 0) + gp
```

**Why these atoms must compose.** The pop-outgrad loop without dispatch
has nothing to call (it knows WHEN to step but not WHAT to invoke).
Dispatch without the loop has no driving traversal (it knows WHAT to
call but not in what ORDER). Together they form the complete reverse-
pass driver — every later autograd extension (gradient checkpointing,
double-backward, mixed precision) is a small modification of THIS loop.

### Composite Exercise — reverse-pass driver: pop next out-grad, dispatch back_fn from recipe

**Atoms exercised together**: `backprop-pop-outgrad-loop`, `dispatch-back-fn-from-recipe`

Implement `cx20_backprop(end_node, end_grad, sorted_graph, back_funcs)` — the full reverse-pass driver. Two atoms compose:

1. **OUTER walk** (`backprop-pop-outgrad-loop`) — iterate `sorted_graph` (end-node FIRST, leaves last), pop each node's accumulated grad from `grads`, route to leaf-write or dispatch.
2. **INNER dispatch** (`dispatch-back-fn-from-recipe`) — for each `(argnum, parent)` in `node.recipe.parents.items()`, look up `back_funcs[(node.recipe.func, argnum)]` and call it with `(grad_out, node.array, *recipe.args, **recipe.kwargs)`.

**Inputs.**
- `end_node` — MiniTensor at which to start the reverse pass.
- `end_grad` — `torch.Tensor` with `dL/d(end_node)`. Usually `t.ones_like`.
- `sorted_graph` — `list[MiniTensor]` in reverse-topological order.
- `back_funcs` — `dict[(forward_fn, argnum), back_fn]` registry.

**Three invariants the test enforces.**
1. **POP, don't peek.** `grads.pop(id(node))` — once popped, the node's grad is gone from the dict.
2. **ACCUMULATE with `+`**, never overwrite — diamond DAGs route grad through the same parent twice.
3. **Leaves write `.grad`; non-leaves stay in `grads`.** Leaf = `node.recipe is None`. For leaves with `.grad` already set, accumulate (don't overwrite).

Return `None`. Mutate `.grad` on each leaf in place.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx20_backprop(end_node, end_grad, sorted_graph, back_funcs):
    """Reverse-pass driver: pop-outgrad loop + recipe-based dispatch."""
    raise NotImplementedError()

def _test_cx20():
    from dataclasses import dataclass, field
    from typing import Any, Callable, Optional

    @dataclass
    class Recipe:
        func: Optional[Callable] = None
        args: tuple = ()
        kwargs: dict = field(default_factory=dict)
        parents: dict = field(default_factory=dict)

    class MiniTensor:
        def __init__(self, array, requires_grad=False, recipe=None):
            self.array = array; self.requires_grad = requires_grad
            self.recipe = recipe; self.grad = None

    # back_fns (raw torch)
    def log_back(grad_out, out, x): return grad_out / x
    def mul_back0(grad_out, out, x, y): return grad_out * y
    def mul_back1(grad_out, out, x, y): return grad_out * x
    BF = {(t.log, 0): log_back, (t.multiply, 0): mul_back0, (t.multiply, 1): mul_back1}

    # === TEST 1: x*y — both leaves get correct grads ===
    x = MiniTensor(t.tensor([2.0, 3.0]), requires_grad=True)
    y = MiniTensor(t.tensor([5.0, 7.0]), requires_grad=True)
    out = MiniTensor(x.array * y.array, requires_grad=True)
    out.recipe = Recipe(func=t.multiply, args=(x.array, y.array), kwargs={}, parents={0: x, 1: y})
    cx20_backprop(out, t.ones(2), [out, x, y], BF)
    assert t.allclose(x.grad, y.array), f'd(xy)/dx=y; got {x.grad}'
    assert t.allclose(y.grad, x.array), f'd(xy)/dy=x; got {y.grad}'

    # === TEST 2: diamond accumulation z * z → d/dz = 2z ===
    z = MiniTensor(t.tensor([3.0]), requires_grad=True)
    out = MiniTensor(z.array * z.array, requires_grad=True)
    out.recipe = Recipe(func=t.multiply, args=(z.array, z.array), kwargs={}, parents={0: z, 1: z})
    cx20_backprop(out, t.ones(1), [out, z], BF)
    assert t.allclose(z.grad, t.tensor([6.0])), f'd(z^2)/dz=2z=6; got {z.grad}'

    # === TEST 3: chain log(b) then a * c — three-node graph ===
    import math
    a = MiniTensor(t.tensor([2.0]), requires_grad=True)
    b = MiniTensor(t.tensor([math.e]), requires_grad=True)
    c = MiniTensor(t.log(b.array), requires_grad=True)
    c.recipe = Recipe(func=t.log, args=(b.array,), kwargs={}, parents={0: b})
    out = MiniTensor(a.array * c.array, requires_grad=True)
    out.recipe = Recipe(func=t.multiply, args=(a.array, c.array), kwargs={}, parents={0: a, 1: c})
    cx20_backprop(out, t.ones(1), [out, a, c, b], BF)
    assert t.allclose(a.grad, t.tensor([1.0]), atol=1e-5), f'a.grad={a.grad}'
    assert t.allclose(b.grad, t.tensor([2.0 / math.e]), atol=1e-5), f'b.grad={b.grad}'

    # === TEST 4: leaf .grad accumulates across calls (does not overwrite) ===
    a = MiniTensor(t.tensor([1.0]), requires_grad=True)
    a.grad = t.tensor([10.0])
    out = MiniTensor(t.log(a.array), requires_grad=True)
    out.recipe = Recipe(func=t.log, args=(a.array,), kwargs={}, parents={0: a})
    cx20_backprop(out, t.ones(1), [out, a], BF)
    assert t.allclose(a.grad, t.tensor([11.0])), f'must accumulate (10+1=11), got {a.grad}'

    # === TEST 5: dispatch via (recipe.func, argnum) — asymmetric mul_back0 vs mul_back1 ===
    # Different x, y values → confirm mul_back0 picked y (not x) and vice versa
    x = MiniTensor(t.tensor([4.0]), requires_grad=True)
    y = MiniTensor(t.tensor([11.0]), requires_grad=True)
    out = MiniTensor(x.array * y.array, requires_grad=True)
    out.recipe = Recipe(func=t.multiply, args=(x.array, y.array), kwargs={}, parents={0: x, 1: y})
    cx20_backprop(out, t.ones(1), [out, x, y], BF)
    assert t.allclose(x.grad, t.tensor([11.0])), f'mul_back0 must pick y=11; got {x.grad}'
    assert t.allclose(y.grad, t.tensor([4.0])), f'mul_back1 must pick x=4; got {y.grad}'
    _dd_passed.add('cx20')

_test_cx20()

<details><summary>Show solution — cx20</summary>

```python
def cx20_backprop(end_node, end_grad, sorted_graph, back_funcs):
    grads = {id(end_node): end_grad}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            continue                                # not reached this pass
        grad_out = grads.pop(nid)                   # POP, don't peek
        if node.recipe is None:                     # leaf — write .grad
            if node.grad is None:
                node.grad = grad_out
            else:
                node.grad = node.grad + grad_out     # rebind, not in-place
            continue
        # non-leaf: dispatch each parent
        for argnum, parent in node.recipe.parents.items():
            back_fn = back_funcs[(node.recipe.func, argnum)]      # dispatch atom
            grad_parent = back_fn(
                grad_out,
                node.array,
                *node.recipe.args,
                **node.recipe.kwargs,
            )
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + grad_parent          # accumulate
    return None
```

**Two atoms, one loop body.** The outer `for node in sorted_graph` is the `backprop-pop-outgrad-loop` skeleton; the inner `for argnum, parent in ...` is the `dispatch-back-fn-from-recipe` step. They're ALWAYS used together — the dispatch makes no sense without the driving traversal.

**Why `grads.pop` and `grads.get(pid, 0) + gp`.** Pop frees the popped node's grad (no leaks, surfaces double-consumption bugs). `.get(pid, 0) + gp` handles BOTH (a) first-time touch of a parent (seed with 0) and (b) diamond accumulation (add to existing entry).

**Leaves vs non-leaves split.** `node.recipe is None` is the leaf marker; leaves get `.grad` written (rebind, NOT `+=`), non-leaves stay in the scratch `grads` dict. Splitting the storage means the scratch dict can be GC'd at function exit while leaf `.grad`s persist for the optimizer.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx20'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx20',
        'subtopics': ["Backprop: backprop pop-outgrad loop", "Backprop: dispatch back fn from recipe"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()